# Biohub - Cell Tracking During Development: 高スコアnotebook解説

- **コンペ**: [Biohub - Cell Tracking During Development](https://www.kaggle.com/competitions/biohub-cell-tracking-during-development)（ゼブラフィッシュの細胞を3D空間・時間方向に検出・追跡する研究コンペ、賞金 $60,000）
- **元notebook**: [Biohub Cell Tracking Solution](https://www.kaggle.com/code/kaiwalyaatulraut/biohub-cell-tracking-solution) by **KAIWALYA RAUT**（154 votes / Gold medal）
- **スコア**: このnotebook自身のページ表示では Public Score **0.885** / Best Score **0.901**（V4時点）。コンペのCodeタブ一覧（Best Scoreソート）では同notebookが **0.966** と表示されており、バージョン間・表示タイミングの差の可能性があります（本ノートは実行済み出力を含まないため断定はできません。数値の食い違いも含めて事実として記録しています）。
- **手法概要**: 3D U-Net（`TemporalUNet3D`）で各フレームの細胞中心らしさ（point probability）を予測し、Transformer（`SimpleNodeTransformer`）で隣接フレーム間のノード対応（エッジ確率）を予測。8方向のTTA（反転・回転）で予測を安定化し、`tracksdata`ライブラリのILP（整数計画法）ソルバーでグラフ全体として整合的なリンクを選択。さらに1フレームだけ検出が欠けた場合のギャップクロージング（Hungarian法）、短すぎるトラックの除去、最後に謎の「合成コンポーネント追加」処理を行っています。

> **お断り**: これは学習目的の解説付き写しです。コード自体の改変（ロジック変更）はしていませんが、Kaggle上のHTML表示からテキストを取得した際にインデント（字下げ）情報が失われたため、**インデントは元のロジックが破綻しないよう書き起こし時に再構成しています**（変数名・関数名・処理の流れ・コメントは元notebookのまま）。出力（実行結果）はコピー元に含めていません。

## 評価指標

- **タスク**: 3D顕微鏡タイムラプス画像中の各フレームで細胞の中心座標（ノード）を検出し、フレーム間で同じ細胞を追跡するエッジ（リンク）を張り、細胞分裂（division）も検出してリネージ（系譜）グラフを再構成する。
- **指標**: notebook内の評価コード（`evaluate`関数）は `edge_tp` / `edge_fp` / `edge_fn`（エッジの正解・過検出・見逃し数）と `division_tp/fp/fn`（分裂イベントの正誤）、`node_recall`（ノード検出の再現率）を算出し、`edge_jaccard` / `adj_edge_jaccard`（エッジ集合のJaccard係数=TP/(TP+FP+FN)）としてまとめています。CTC（Cell Tracking Challenge）系のトラッキング評価でよく使われる、**検出（ノード）と時系列リンク（エッジ）の両方をグラフとして採点する指標**です。
- **なぜこの指標か**: 単純な「細胞が何個検出できたか」だけでは、フレーム間の対応関係（どの細胞がどの細胞に成長したか、分裂したか）という追跡タスク本来の難しさを評価できません。エッジ単位でTP/FP/FNを数えるJaccard型の指標は、検出精度と時系列の整合性を同時に罰則・評価できます。位置ずれには`max_distance`（許容誤差）を設けることで、多少のピクセル単位のずれは許容しつつ、トポロジー（つながり方）の正しさを重視する設計です。
- **この手法がどう指標を最適化しているか**: (1) U-Netで検出漏れを減らすためTTA（8方向flip/rotation平均）で予測を安定化、(2) Transformerでエッジ確率を出し、ILPソルバーで「全体として矛盾のないグラフ」になるよう厳密最適化（貪欲法ではなく大域最適に近い解を求める）、(3) 後処理で1フレームだけ検出が飛んだ場合の穴埋め（gap closing）を行い`edge_fn`（見逃し)を減らす、(4) 短い孤立トラックを除去してノイズによる`edge_fp`を抑える、という流れで指標の各要素（TP↑, FP↓, FN↓）を意識した設計になっています。


In [ ]:
import os, json, glob, time
from collections import Counter

import numpy as np
import pandas as pd
import blosc2
from scipy.ndimage import gaussian_filter
from scipy.optimize import linear_sum_assignment
from skimage.feature import peak_local_max
from skimage.filters import threshold_otsu

SCALE = np.array([1.625, 0.40625, 0.40625])

XY_DS = 4
SMOOTH_SIGMA = 1.0
MIN_PEAK_DIST = 3
THRESH_REL = 0.30

MAX_LINK_DIST = 12.0

DETECT_DIVISIONS = True
DIV_PARENT_DIST = 12.0
DIV_SISTER_DIST = 7.0

CANDIDATES = [
    '/kaggle/input/competitions/biohub-cell-tracking-during-development/test',
    '/kaggle/input/biohub-cell-tracking-during-development/test',
]
TEST_DIR = next((p for p in CANDIDATES if os.path.isdir(p)), None)
if TEST_DIR is None:
    hits = glob.glob('/kaggle/input/**/test', recursive=True)
    TEST_DIR = next((h for h in hits if glob.glob(os.path.join(h, '*.zarr'))), hits[0] if hits else CANDIDATES[0])
print('TEST_DIR =', TEST_DIR)


**何をしているか**: 定数（スケール、閾値、リンク距離など）を定義し、テストデータのディレクトリを探索している。
**なぜそうするのか**: Kaggle環境では入力データのマウントパスがコンペごと・実行環境ごとに微妙に変わることがあるため、候補パスを順に試し、見つからなければ`glob`で総当たり探索するという防御的な書き方をしている（初心者向け補足: `glob`はワイルドカード（`**`など）でファイル・フォルダを検索する標準ライブラリ）。

In [ ]:
try:
    import zarr, geff, tracksdata
except Exception:
    import os
    import sys
    import subprocess
    import importlib
    import importlib.util
    from pathlib import Path

    os.environ.setdefault("POLARS_PREFER_PKG", "32")

    SUPPORT_DIR = Path(
        "/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1"
    )
    WHEELS_DIR = SUPPORT_DIR / "wheels"

    if not WHEELS_DIR.exists():
        candidates = list(Path("/kaggle/input").glob("**/wheels"))
        if not candidates:
            raise FileNotFoundError("Could not find the attached offline wheels directory")
        WHEELS_DIR = candidates[0]

    print("Offline wheels:", WHEELS_DIR)

    OFFLINE_PACKAGES = [
        "tracksdata",
        "zarr==3.2.1",
        "numcodecs==0.15.1",
        "donfig==0.8.1.post1",
        "geff==1.2.0.1.1",
        "geff-spec==1.1.1",
        "pyscipopt==6.2.1",
        "ilpy==0.6.0",
        "rustworkx==0.18.0",
        "polars==1.42.0",
        "polars-runtime-32==1.42.0",
        "bidict==0.23.1",
        "imagecodecs==2026.6.26",
    ]

    def module_missing(module_name: str) -> bool:
        return importlib.util.find_spec(module_name) is None

    REQUIRED_IMPORTS = {
        "tracksdata": "tracksdata",
        "zarr": "zarr",
        "numcodecs": "numcodecs",
        "geff": "geff",
        "pyscipopt": "pyscipopt",
        "ilpy": "ilpy",
        "rustworkx": "rustworkx",
        "polars": "polars",
        "imagecodecs": "imagecodecs",
    }

    def purge_modules(module_roots):
        """
        Remove already-imported package modules from sys.modules.
        Normally this cell runs before imports, but this also protects against
        accidental imports performed by earlier Kaggle initialization code.
        """
        for root in module_roots:
            for name in list(sys.modules):
                if name == root or name.startswith(root + "."):
                    sys.modules.pop(name, None)

    def install_offline_packages():
        cmd = [
            sys.executable, "-m", "pip", "install", "--quiet",
            "--no-index", "--no-deps", "--find-links", str(WHEELS_DIR),
            *OFFLINE_PACKAGES,
        ]
        print("Installing attached packages without modifying NumPy/SciPy/Torch...")
        result = subprocess.run(cmd, text=True, capture_output=True)
        if result.returncode != 0:
            print(result.stdout[-4000:])
            print(result.stderr[-4000:])
            raise RuntimeError("Offline dependency installation failed")

    purge_modules(REQUIRED_IMPORTS.values())
    install_offline_packages()

    failures = {}
    for name, module_name in {
        **REQUIRED_IMPORTS, "numpy": "numpy", "scipy": "scipy",
        "dask": "dask.array", "xarray": "xarray",
    }.items():
        try:
            importlib.import_module(module_name)
        except Exception as exc:
            failures[name] = f"{type(exc).__name__}: {exc}"

    if failures:
        raise ImportError(
            "Dependency verification failed:\n"
            + "\n".join(f"{name}: {error}" for name, error in failures.items())
        )

    import zarr, geff, tracksdata
    print("All offline dependencies imported successfully.")


**何をしているか**: `zarr`（大規模配列をチャンク単位で扱うフォーマット）、`geff`（グラフ交換フォーマット）、`tracksdata`（トラッキング用グラフライブラリ）が使えるか試し、使えなければKaggle上に添付された「オフラインwheel（ビルド済みパッケージファイル）」からインターネット接続なしでインストールする。
**なぜそうするのか**: Code Competitionの推論環境はインターネットアクセスが無効化されているため、`pip install`で外部サーバーから取得することができない。そこで主催者・投稿者があらかじめ`.whl`ファイル一式をKaggle Datasetとして添付しておき、`--no-index --find-links`（PyPIを見ずにローカルのwheelフォルダだけを見る）でオフラインインストールするのが定番パターン（初心者向け補足: `--no-deps`は依存関係を自動解決させず、必要なパッケージだけを明示的に入れることで、NumPy/PyTorchなど既存環境のバージョンを壊さないようにする工夫）。

In [ ]:
import sys
sys.path.append("/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1/repo/src")

from biohub_tracking.models import TemporalUNet3D, SimpleNodeTransformer
from biohub_tracking.io import open_dataset, save_graph

import os
import contextlib
import zarr
import numpy as np
from tqdm import tqdm
import json
import glob
import csv
import pandas as pd
from joblib import Parallel, delayed

import torch
import torch.nn as nn
import torch.nn.functional as F

import tracksdata as td
import polars as pl
import pandas as pd

from geff import GeffMetadata
from biohub_tracking.metrics import (
    evaluate,
    node_recall,
    per_sample_metrics,
    summarise,
)

MODE = "submit"

KAGGLE_DIR = "/kaggle/input/competitions/biohub-cell-tracking-during-development"
if MODE == "local":
    valid_id = ['44b6_0113de3b', '44b6_0b24845f', '6bba_05b6850b', '6bba_05db0fb1', '44b6_33b596bf']
    valid_dir = "/kaggle/input/competitions/biohub-cell-tracking-during-development/train"

if MODE == "submit":
    glob_file = glob.glob(f"/kaggle/input/competitions/biohub-cell-tracking-during-development/test/*.zarr")
    valid_id = sorted([f.split("/")[-1][:-5] for f in glob_file])
    valid_dir = "/kaggle/input/competitions/biohub-cell-tracking-during-development/test"

print("MODE:", MODE)
print("valid_id:", len(valid_id), valid_id[:4])
print("setup ok!!!!!")


**何をしているか**: 主催者提供の学習済みモデルクラス（`TemporalUNet3D`, `SimpleNodeTransformer`）と評価関数（`evaluate`, `node_recall`など）をインポートし、実行モードを`"submit"`（本番のtestデータに対して推論）に設定して対象サンプルID一覧を取得する。
**なぜそうするのか**: `MODE`で`"local"`（trainデータで手元検証）と`"submit"`（testデータで実際に提出用ファイルを作る）を切り替えられるようにしておくことで、同じコードをローカル検証と本番提出の両方に使い回せる（初心者向け補足: こうした「モード切り替え変数」はKaggle notebookでの検証・提出コード共通化の定番パターン）。

In [ ]:
DEVICE = "cuda"
SUBSAMPLE = [1, 4, 4]
VOLUME_SHAPE = [64, 64, 64]
TIME_LENGTH = 2

POINT_THRESHOLD = 0.9500
USE_TTA = True
USE_MULTI_GPU = True

ILP_EDGE_WEIGHT = -1.0
ILP_APPEARANCE_WEIGHT = 0.0
ILP_DISAPPEARANCE_WEIGHT = 1.4
ILP_DIVISION_WEIGHT = 1.0

EDGE_STRONG_THRESHOLD = 0.50
EDGE_MIN_THRESHOLD = 0.25
EDGE_TOPK_PARENTS = 3

EDGE_MAX_DISTANCE_UM = 10.0


class MyUnet(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.D = nn.Parameter(torch.ones(1))

        self.unet = TemporalUNet3D(
            in_channels=1,
            out_channels=int(config["unet_out_channels"]),
            layers=tuple(config["unet_layers"]),
            gradient_checkpointing=False,
        )
        unet_out_channels = int(config["unet_out_channels"])
        self.unet_out_channels = unet_out_channels
        self.detect_head = nn.Conv3d(unet_out_channels, 1, kernel_size=1)

        pos_feat_dim = 4 * 8
        self.transformer = SimpleNodeTransformer(
            feat_dim=unet_out_channels + pos_feat_dim,
            hidden_dim=128, n_heads=4, n_blocks=4, dropout=0,
        )

    def forward_unet(self, image: torch.Tensor):
        image = image[:, :, None]
        f = self.unet(image)
        point_logit = [self.detect_head(f[:, 0]), self.detect_head(f[:, 1])]
        point_feature = [f[:, 0], f[:, 1]]
        return point_feature, point_logit

    def forward_transformer(self, select0, select1, coord0, coord1, pos0, pos1):
        feature0 = torch.cat([select0, pos0], dim=-1)
        feature1 = torch.cat([select1, pos1], dim=-1)
        logit = self.transformer(feature0, feature1, coord0, coord1)
        return logit


def embed_position(zyx, t, image_shape=VOLUME_SHAPE, time_length=TIME_LENGTH, pos_per_dim=8):
    zyx = zyx.float()
    z, y, x = zyx.unbind(dim=1)
    t_tensor = torch.as_tensor(t, dtype=zyx.dtype, device=zyx.device)
    t_normalized = torch.ones_like(z) * (t_tensor / time_length)
    tzyx = [t_normalized, z / image_shape[0], y / image_shape[1], x / image_shape[2]]

    def embed(values):
        freqs = 2.0 ** torch.arange(pos_per_dim // 2, dtype=values.dtype, device=values.device)
        angles = values[:, None] * freqs[None, :] * torch.pi
        return torch.cat([torch.sin(angles), torch.cos(angles)], dim=1)

    return torch.cat([embed(values) for values in tzyx], dim=1)


def pool_kernel_from_um(um, voxel_size):
    kernel = []
    for s in voxel_size:
        k = max(1, round(um / s))
        if k % 2 == 0:
            k += 1
        kernel.append(k)
    return tuple(kernel)


def prob_to_zyx(prob, threshold=0.5, pool_kernel=(3, 3, 3)):
    prob = prob.unsqueeze(0)
    pad = tuple(k // 2 for k in pool_kernel)
    pooled = F.max_pool3d(prob, pool_kernel, stride=1, padding=pad)
    is_peak = (prob == pooled) & (prob > threshold)
    peak_idx = torch.nonzero(is_peak[0, 0])
    if peak_idx.shape[0] == 0:
        return torch.empty((0, 3), dtype=torch.long)
    return peak_idx


def select_feature(feature, zyx):
    _, Z, Y, X = feature.shape
    z = zyx[:, 0].long().clamp(0, Z - 1)
    y = zyx[:, 1].long().clamp(0, Y - 1)
    x = zyx[:, 2].long().clamp(0, X - 1)
    selected = feature[:, z, y, x]
    return selected.permute(1, 0).contiguous()


def build_graph(coord, edge):
    graph = td.graph.InMemoryGraph()
    for key in ["z", "y", "x"]:
        graph.add_node_attr_key(key, pl.Float64, -999999.0)

    node_ids = graph.bulk_add_nodes([
        {"t": int(t), "z": float(z), "y": float(y), "x": float(x)}
        for t, z, y, x in coord
    ])

    if edge:
        graph.add_edge_attr_key("edge_prob", pl.Float64, 0.0)
        graph.add_edge_attr_key("edge_dist", pl.Float64, 0.0)
        graph.bulk_add_edges([
            {"source_id": node_ids[i], "target_id": node_ids[j], "edge_prob": prob, "edge_dist": dist}
            for i, j, prob, dist in edge
        ])
    return graph


def load_model_weight(weight_file, model):
    state = torch.load(weight_file, map_location="cpu", weights_only=True)
    missing, unexpected = model.load_state_dict(state, strict=False)
    print(f"loaded weight: {weight_file}")
    print(f"\tmissing key: {len(missing)}", missing)
    print(f"\tunexpected key: {len(unexpected)}", unexpected)
    return model


def load_volume(sample_id):
    zarr_file = f"{valid_dir}/{sample_id}.zarr"
    ds = open_dataset(zarr_file, normalize=False, load_image=False, require_tracks=False)

    zarr_arr = zarr.open_group(str(ds.zarr_path), mode="r")["0"]
    q_low = float(ds.quantiles["0.001"])
    q_high = float(ds.quantiles["0.999"])
    dz, dy, dx = SUBSAMPLE
    small = zarr_arr[:, ::dz, ::dy, ::dx].astype(np.float32)
    assert small.shape[1:] == tuple(VOLUME_SHAPE)

    small = (small - q_low) / (q_high - q_low + 1e-6)
    small = np.clip(small, 0.0, None)
    voxel_size = tuple(s * d for s, d in zip(ds.scale, SUBSAMPLE))
    meta = {"voxel_size": voxel_size}
    return small, meta


# --- Test-Time Augmentation (TTA) の3種類のバリエーション ---
# 上下左右flip、90度回転の組み合わせで推論を複数回行い平均することでノイズに強くする。

def do_tta_4flip(im):
    image = [im]
    image += [im.flip(dims=(2,))]
    image += [im.flip(dims=(3,))]
    image += [im.flip(dims=(2, 3))]
    return image, None


def undo_tta_4flip(x, transform=None):
    x[0] = x[0]
    x[1] = x[1].flip(dims=(2,))
    x[2] = x[2].flip(dims=(3,))
    x[3] = x[3].flip(dims=(2, 3))
    return x


def do_tta_8yx(im):
    image, transform = [], []
    for flip_x in (False, True):
        for k in range(4):
            x = im
            if flip_x:
                x = x.flip(dims=(-1,))
            x = torch.rot90(x, k=k, dims=(-2, -1))
            image.append(x)
            transform.append((k, flip_x))
    return image, transform


def undo_tta_8yx(x, transform):
    restored = []
    for i in range(len(transform)):
        k, flip_x = transform[i]
        xi = x[i]
        xi = torch.rot90(xi, k=(-k) % 4, dims=(-2, -1))
        if flip_x:
            xi = xi.flip(dims=(-1,))
        restored.append(xi)
    return torch.stack(restored, dim=0)


def do_tta_8fliprot(im):
    dims = (-2, -1)
    images = [
        im, im.flip(dims=(-1,)), im.flip(dims=(-2,)), im.flip(dims=(-2, -1)),
        torch.rot90(im, 1, dims=dims), torch.rot90(im, 3, dims=dims),
        im.transpose(-1, -2), torch.rot90(im, 1, dims=dims).transpose(-1, -2),
    ]
    return images, None


def undo_tta_8fliprot(x, transform=None):
    dims = (-2, -1)
    return torch.stack([
        x[0], x[1].flip(dims=(-1,)), x[2].flip(dims=(-2,)), x[3].flip(dims=(-2, -1)),
        torch.rot90(x[4], -1, dims=dims), torch.rot90(x[5], -3, dims=dims),
        x[6].transpose(-1, -2), torch.rot90(x[7].transpose(-1, -2), -1, dims=dims),
    ])


def do_tta_9public(im):
    dims = (-2, -1)
    images = [
        im, im.flip(dims=(-1,)), im.flip(dims=(-2,)), im.flip(dims=(-2, -1)),
        im.rot90(1, dims=dims), im.rot90(2, dims=dims), im.rot90(3, dims=dims),
        im.transpose(-1, -2), im.rot90(1, dims=dims).transpose(-1, -2),
    ]
    return images, None


def undo_tta_9public(x, transform=None):
    dims = (-2, -1)
    return torch.stack([
        x[0], x[1].flip(dims=(-1,)), x[2].flip(dims=(-2,)), x[3].flip(dims=(-2, -1)),
        x[4].rot90(-1, dims=dims), x[5].rot90(-2, dims=dims), x[6].rot90(-3, dims=dims),
        x[7].transpose(-1, -2), x[8].transpose(-1, -2).rot90(-1, dims=dims),
    ])


@contextlib.contextmanager
def suppress_output():
    """Context manager to suppress stdout and stderr."""
    with open(os.devnull, "w") as devnull:
        with contextlib.redirect_stdout(devnull), contextlib.redirect_stderr(devnull):
            yield


def predict_one(model, volume, meta):
    point_threshold = POINT_THRESHOLD
    pool_kernel_um = 3.0

    device = model.D.device
    voxel_size = meta["voxel_size"]
    pool_kernel = pool_kernel_from_um(pool_kernel_um, voxel_size)
    subsample = torch.as_tensor([SUBSAMPLE], dtype=torch.float32, device=device)

    raw_voxel_size = np.asarray(voxel_size, dtype=np.float64) / np.asarray(SUBSAMPLE, dtype=np.float64)
    T = volume.shape[0]

    out_edge, out_node, out_start = [], [], {}

    def add_to_out(edge_prob, coord0, coord1, t0, t1):
        if t0 == 0:
            out_start[t0] = len(out_node)
        for z, y, x in coord0:
            out_node.append([t0, z, y, x])
        out_start[t1] = len(out_node)
        for z, y, x in coord1:
            out_node.append([t1, z, y, x])

        number_source, number_target = edge_prob.shape
        if number_source == 0 or number_target == 0:
            return

        candidate_pairs = set()
        strong_indices = np.argwhere(edge_prob >= EDGE_STRONG_THRESHOLD)
        for source_index, target_index in strong_indices:
            candidate_pairs.add((int(source_index), int(target_index)))

        top_k = min(EDGE_TOPK_PARENTS, number_source)
        for target_index in range(number_target):
            probabilities = edge_prob[:, target_index]
            if top_k == number_source:
                top_source_indices = np.arange(number_source)
            else:
                top_source_indices = np.argpartition(probabilities, -top_k)[-top_k:]
            for source_index in top_source_indices:
                probability = float(probabilities[source_index])
                if probability < EDGE_MIN_THRESHOLD:
                    continue
                candidate_pairs.add((int(source_index), int(target_index)))

        candidates = []
        for source_index, target_index in candidate_pairs:
            source_position = coord0[source_index]
            target_position = coord1[target_index]
            delta_um = (source_position - target_position) * raw_voxel_size
            distance_um = float(np.linalg.norm(delta_um))
            if distance_um > EDGE_MAX_DISTANCE_UM:
                continue
            probability = float(edge_prob[source_index, target_index])
            candidates.append((probability, source_index, target_index, distance_um))

        candidates.sort(reverse=True)
        start0, start1 = out_start[t0], out_start[t1]
        for probability, source_index, target_index, distance_um in candidates:
            out_edge.append([source_index + start0, target_index + start1, probability, distance_um])

    for t in tqdm(range(T - 1), total=T - 1, leave=False, disable=False):
        im = torch.from_numpy(volume[t:t + 2]).to(device)
        image = [im]

        with torch.inference_mode():
            if USE_TTA:
                do_tta, undo_tta = do_tta_8fliprot, undo_tta_8fliprot
                image, transform = do_tta(im)

            image = torch.stack(image, dim=0)
            point_feature, point_logit = model.forward_unet(image)

            if USE_TTA:
                point_feature = [undo_tta(x, transform) for x in point_feature]
                point_logit = [undo_tta(x, transform) for x in point_logit]

            point_prob = [torch.sigmoid(x.mean(0)) for x in point_logit]

            if USE_TTA:
                edge_feature0 = point_feature[0].mean(dim=0)
                edge_feature1 = point_feature[1].mean(dim=0)
            else:
                edge_feature0 = point_feature[0][0]
                edge_feature1 = point_feature[1][0]

            if t == 0:
                zyx0 = prob_to_zyx(point_prob[0], pool_kernel=pool_kernel, threshold=point_threshold)
            else:
                zyx0 = zyx1

            pos0 = embed_position(zyx0, t=0, pos_per_dim=8)
            coord0 = zyx0 * subsample
            select0 = select_feature(edge_feature0, zyx0).unsqueeze(0)
            zyx1 = prob_to_zyx(point_prob[1], pool_kernel=pool_kernel, threshold=point_threshold)
            pos1 = embed_position(zyx1, t=1, pos_per_dim=8)
            coord1 = zyx1 * subsample
            select1 = select_feature(edge_feature1, zyx1).unsqueeze(0)

            E = len(select0)
            edge_logit = model.forward_transformer(
                select0, select1,
                coord0[None].expand(E, -1, -1), coord1[None].expand(E, -1, -1),
                pos0[None].expand(E, -1, -1), pos1[None].expand(E, -1, -1),
            )
            edge_prob = torch.softmax(edge_logit.mean(0), dim=0)

            add_to_out(
                edge_prob.float().data.cpu().numpy(),
                coord0.float().data.cpu().numpy(),
                coord1.float().data.cpu().numpy(),
                t0=t, t1=t + 1,
            )

    return out_node, out_edge


print("modeling ok !!!")


**何をしているか**: モデル本体（3D U-Net + Transformer）の定義、TTA（Test-Time Augmentation：推論時に画像を反転・回転させた複数バージョンで予測し平均する手法）の4パターンの実装、そして1フレーム分の推論を行う`predict_one`関数をまとめて定義している、このnotebookの中核部分。
**なぜそうするのか**:
- `embed_position`は座標を正弦・余弦の波（sin/cos）に変換する「位置エンコーディング」（Transformer系モデルの定番手法）。座標をそのまま数値として渡すより、周期関数に変換した方はモデルが位置関係を学習しやすいことが経験的に知られている（初心者向け補足: これはTransformerの原論文でも使われている手法で、単純な整数の座標よりも学習が安定しやすい）。
- `prob_to_zyx`は「極大値検出（non-maximum suppression, NMS）」。3D空間で近傍より確率が高い点だけを細胞の中心候補として残すことで、1つの細胞に対して重複した検出が出るのを防ぐ。
- TTAを4種類実装しているのは、どのパターンが最も精度に寄与するか試行錯誤した跡（`do_tta_4flip`, `do_tta_8yx`, `do_tta_8fliprot`, `do_tta_9public`）。実際に使われているのは`predict_one`内で選ばれている`do_tta_8fliprot`（8方向の反転・回転・転置）。
- `predict_one`のロジックは「候補ペアの絞り込み」が肝：まず確率が高い強いペアを全部残し（`EDGE_STRONG_THRESHOLD`）、次に各ターゲットについて上位k件の候補元ノードを追加（`EDGE_TOPK_PARENTS`）することで、計算量を抑えつつ見逃し（FN）を減らす設計になっている。

In [ ]:
checkpoint_dir = \
    "/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1/weights/unet_transformer/split_0"

checkpoint_file = f"{checkpoint_dir}/edge_predictor_best.pth"
config_file = f"{checkpoint_dir}/config.json"

predict_dir = "/kaggle/working/my_predict"
os.makedirs(predict_dir, exist_ok=True)


def run_worker(gpu_id: int, subset_id):
    torch.cuda.set_device(gpu_id)
    device = torch.device(f"cuda:{gpu_id}")

    with open(config_file, "r", encoding="utf-8") as f:
        config = json.load(f)

    model = MyUnet(config)
    load_model_weight(checkpoint_file, model)
    model.to(device)
    model.eval()

    for sample_id in subset_id:
        volume, meta = load_volume(sample_id)
        out_node, out_edge = predict_one(model, volume, meta)
        graph = build_graph(out_node, out_edge)

        if graph.num_edges() > 0:
            solver = td.solvers.ILPSolver(
                edge_weight=ILP_EDGE_WEIGHT * td.EdgeAttr("edge_prob"),
                appearance_weight=ILP_APPEARANCE_WEIGHT,
                disappearance_weight=ILP_DISAPPEARANCE_WEIGHT,
                division_weight=ILP_DIVISION_WEIGHT,
                num_threads=1,
            )
            graph = solver.solve(graph)

        save_graph(graph, f"{predict_dir}/{sample_id}.geff")
    del model
    torch.cuda.empty_cache()
    return gpu_id


if USE_MULTI_GPU:
    subset_id0 = valid_id[0::2]
    subset_id1 = valid_id[1::2]

    result = Parallel(n_jobs=2, backend="loky", verbose=10)(
        [delayed(run_worker)(0, subset_id0), delayed(run_worker)(1, subset_id1)]
    )
    print(result)
else:
    run_worker(0, valid_id)


**何をしているか**: 学習済み重み（`edge_predictor_best.pth`）を読み込み、`run_worker`関数で「モデル推論 → グラフ構築 → ILPで最適なリンクを選択 → geffファイルに保存」という一連の処理をサンプルごとに実行。2GPU環境ではサンプルを偶数番目・奇数番目に分けて並列実行している。
**なぜそうするのか**: ここで初めて`ILPSolver`（整数線形計画法ソルバー）が登場する。ノードごと・エッジごとに「出現」「消失」「分裂」に重みを設定し（`ILP_APPEARANCE_WEIGHT`など）、グラフ全体として最も辻褄が合う（矛盾のない）リンクの組み合わせを厳密に解く。貪欲法（近いものから順にリンクを確定する方法）と違い、大域的に最適な解を求められるのが強み（初心者向け補足: ILP=Integer Linear Programming。制約付き最適化問題を厳密に解く手法で、実務では組合せ最適化・スケジューリングにもよく使われる）。GPUを2枚使った並列化（`joblib.Parallel`）は単純に処理時間短縮のため。

In [ ]:
SUBMISSION_PATH = "submission.csv"
SUBMISSION_COLUMN = ["id", "dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"]

glob_file = glob.glob(f"{predict_dir}/*.geff")
print(f"predict_dir: {len(glob_file)}")

row_id = 0
total_num_node = 0
total_num_edge = 0

with open(SUBMISSION_PATH, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=SUBMISSION_COLUMN)
    writer.writeheader()

    for sample_id in valid_id:
        dataset = sample_id
        graph = td.graph.IndexedRXGraph.from_geff(f"{predict_dir}/{sample_id}.geff")[0]

        node_row = list(graph.node_attrs().iter_rows(named=True))
        edge_row = list(graph.edge_attrs().iter_rows(named=True))

        node_id = {int(row["node_id"]) for row in node_row}
        if not node_id:
            raise AssertionError(f"{dataset}: ILP graph contains no nodes")

        for row in sorted(node_row, key=lambda x: int(x["node_id"])):
            writer.writerow({
                "id": row_id, "dataset": dataset, "row_type": "node",
                "node_id": int(row["node_id"]), "t": int(row["t"]),
                "z": max(0, int(round(float(row["z"])))),
                "y": max(0, int(round(float(row["y"])))),
                "x": max(0, int(round(float(row["x"])))),
                "source_id": -1, "target_id": -1,
            })
            row_id += 1

        for row in edge_row:
            source_id = int(row["source_id"])
            target_id = int(row["target_id"])
            if source_id not in node_id or target_id not in node_id:
                raise AssertionError(f"{dataset}: dangling ILP edge {source_id}->{target_id}")
            writer.writerow({
                "id": row_id, "dataset": dataset, "row_type": "edge",
                "node_id": -1, "t": -1, "z": -1, "y": -1, "x": -1,
                "source_id": source_id, "target_id": target_id,
            })
            row_id += 1

        total_num_node += len(node_row)
        total_num_edge += len(edge_row)

submit_df = pd.read_csv(SUBMISSION_PATH, nrows=10)
print(submit_df)
print()
print("total_num_node:", total_num_node)
print("total_num_edge:", total_num_edge)
print("submission ok !!!")


**何をしているか**: ILPソルバーで確定したグラフ（`.geff`ファイル）を、コンペ指定の提出フォーマット（ノード行・エッジ行が混在するロング形式のCSV）に変換する。
**なぜそうするのか**: 多くのコンペでは「モデルの内部表現」と「提出ファイルのフォーマット」が異なるため、変換処理が必須になる。ここでは1行が「ノード（細胞の位置）」か「エッジ（フレーム間のリンク）」かを`row_type`列で区別する設計（初心者向け補足: こうした「ロング形式」＝1種類のテーブルに異なる種類のレコードを混在させる設計は、グラフ構造をCSVのような表形式で表現する際によく使われる）。

### ⚠️ 学習者向けの注意（このセルについて）

次のセル（ギャップクロージング＝隙間埋め処理）とその次の augment_dataset セルは、**指標を意識した後処理**の実例です。特に augment_dataset セルは、実際のゼブラフィッシュ追跡とは関係のない「ダミーのノード・エッジ」をサブミッションに追加しており、コンペの採点スクリプトの挙動を突いた**グレーな最適化（いわゆる"metric hack"）の可能性がある**点に注意してください。このコンペのDiscussionには実際に"Metric_hack_last_call"のようなタイトルのnotebookが複数投稿されており、同種のテクニックが話題になっていたことが伺えます。中身を鵜呑みにせず、「なぜこの処理でスコアが上がるのか」を自分で考えながら読むことをおすすめします。

In [ ]:
from pathlib import Path
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist

import numpy as np
import pandas as pd
import zarr

RAW_SUBMISSION_PATH = "submission.csv"
CLEAN_SUBMISSION_PATH = "submission_clean.csv"

VOXEL_SIZE_UM = np.asarray([1.625, 0.40625, 0.40625], dtype=np.float64)

GAP_CLOSE_MAX_TOTAL_UM = 10.0
GAP_REUSE_EXISTING_UM = 2.5
GAP_MAX_ADDED_FRAC = 0.05
GAP_MAX_ADDED_ABS = 1000
GAP_REFINE_WIN_Z = 1
GAP_REFINE_WIN_YX = 4
GAP_REFINE_MAX_SHIFT_UM = 2.5

MIN_TRACK_LEN = 3
KEEP_DIVISION_COMPONENTS = True
KEEP_BOUNDARY_COMPONENTS = True
BOUNDARY_MARGIN_T = 2


def open_image_array(dataset):
    """画像Zarr配列 (T, Z, Y, X) を開く。"""
    zarr_path = Path(valid_dir) / f"{dataset}.zarr"
    if not zarr_path.exists():
        raise FileNotFoundError(f"image file not found: {zarr_path}")
    root = zarr.open(str(zarr_path), mode="r")
    if hasattr(root, "shape") and len(root.shape) == 4:
        return root
    return root["0"]


def distance_um(point_a, point_b):
    """z,y,x座標間の物理距離。"""
    point_a = np.asarray(point_a, dtype=np.float64)
    point_b = np.asarray(point_b, dtype=np.float64)
    return float(np.linalg.norm((point_a - point_b) * VOXEL_SIZE_UM))


def refine_synthetic_node(frame, midpoint):
    """合成ノードの座標を、幾何学的な中点から画像の輝度重心へわずかに寄せる。"""
    midpoint = np.asarray(midpoint, dtype=np.float64)
    center = np.rint(midpoint).astype(int)
    z, y, x = center.tolist()

    z0, z1 = max(0, z - GAP_REFINE_WIN_Z), min(frame.shape[0], z + GAP_REFINE_WIN_Z + 1)
    y0, y1 = max(0, y - GAP_REFINE_WIN_YX), min(frame.shape[1], y + GAP_REFINE_WIN_YX + 1)
    x0, x1 = max(0, x - GAP_REFINE_WIN_YX), min(frame.shape[2], x + GAP_REFINE_WIN_YX + 1)

    patch = np.asarray(frame[z0:z1, y0:y1, x0:x1], dtype=np.float64)
    if patch.size == 0:
        return midpoint

    background = float(np.percentile(patch, 20))
    weights = np.maximum(patch - background, 0)
    weight_sum = float(weights.sum())
    if not np.isfinite(weight_sum) or weight_sum <= 0:
        return midpoint

    zz = np.arange(z0, z1, dtype=np.float64)[:, None, None]
    yy = np.arange(y0, y1, dtype=np.float64)[None, :, None]
    xx = np.arange(x0, x1, dtype=np.float64)[None, None, :]

    refined = np.asarray([
        float((weights * zz).sum() / weight_sum),
        float((weights * yy).sum() / weight_sum),
        float((weights * xx).sum() / weight_sum),
    ], dtype=np.float64)

    shift_um = distance_um(midpoint, refined)
    if not np.isfinite(shift_um) or shift_um > GAP_REFINE_MAX_SHIFT_UM:
        return midpoint
    return refined


def build_degrees(node_ids, edges):
    """各ノードの入次数・出次数を数える。"""
    in_degree = {int(n): 0 for n in node_ids}
    out_degree = {int(n): 0 for n in node_ids}
    for source_id, target_id in edges:
        source_id, target_id = int(source_id), int(target_id)
        if source_id in out_degree:
            out_degree[source_id] += 1
        if target_id in in_degree:
            in_degree[target_id] += 1
    return in_degree, out_degree


def find_components(node_ids, edges):
    """Union-Findで弱連結成分（エッジの向きを無視した連結グループ）を求める。"""
    node_ids = [int(n) for n in node_ids]
    parent = {n: n for n in node_ids}
    rank = {n: 0 for n in node_ids}

    def find(node_id):
        while parent[node_id] != node_id:
            parent[node_id] = parent[parent[node_id]]
            node_id = parent[node_id]
        return node_id

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra == rb:
            return
        if rank[ra] < rank[rb]:
            parent[ra] = rb
        elif rank[ra] > rank[rb]:
            parent[rb] = ra
        else:
            parent[rb] = ra
            rank[ra] += 1

    for source_id, target_id in edges:
        source_id, target_id = int(source_id), int(target_id)
        if source_id in parent and target_id in parent:
            union(source_id, target_id)

    components = {}
    for node_id in node_ids:
        root = find(node_id)
        components.setdefault(root, []).append(node_id)
    return list(components.values())


def postprocess_one_dataset(group):
    """1. ちょうど1フレーム分の隙間を埋める。 2. 残った短すぎる連結成分を除去する。"""
    dataset = str(group["dataset"].iloc[0])
    image_array = open_image_array(dataset)
    T, Z, Y, X = map(int, image_array.shape)

    node_frame = group[group["row_type"] == "node"][["node_id", "t", "z", "y", "x"]].copy()
    edge_frame = group[group["row_type"] == "edge"][["source_id", "target_id"]].copy()

    nodes = {}
    for row_data in node_frame.itertuples(index=False):
        node_id = int(row_data.node_id)
        nodes[node_id] = {"node_id": node_id, "t": int(row_data.t),
                           "z": float(row_data.z), "y": float(row_data.y), "x": float(row_data.x)}

    edges = {(int(r.source_id), int(r.target_id)) for r in edge_frame.itertuples(index=False)}
    for source_id, target_id in edges:
        if source_id not in nodes or target_id not in nodes:
            raise ValueError(f"{dataset}: dangling edge {source_id}->{target_id}")

    nodes_at_time = {}
    for node_id, node in nodes.items():
        nodes_at_time.setdefault(int(node["t"]), []).append(node_id)

    in_degree, out_degree = build_degrees(nodes.keys(), edges)
    next_node_id = max(nodes) + 1 if nodes else 0
    max_gap_pairs = min(GAP_MAX_ADDED_ABS, max(1, int(len(nodes) * GAP_MAX_ADDED_FRAC)))

    gap_pairs_added = 0
    reused_existing = 0
    synthetic_added = 0

    for source_t in range(0, T - 2):
        if gap_pairs_added >= max_gap_pairs:
            break
        target_t = source_t + 2
        middle_t = source_t + 1

        source_ids = [n for n in nodes_at_time.get(source_t, []) if out_degree.get(n, 0) == 0]
        target_ids = [n for n in nodes_at_time.get(target_t, []) if in_degree.get(n, 0) == 0]
        if not source_ids or not target_ids:
            continue

        source_coordinates = np.asarray(
            [[nodes[n]["z"], nodes[n]["y"], nodes[n]["x"]] for n in source_ids], dtype=np.float64)
        target_coordinates = np.asarray(
            [[nodes[n]["z"], nodes[n]["y"], nodes[n]["x"]] for n in target_ids], dtype=np.float64)

        cost_matrix = cdist(source_coordinates * VOXEL_SIZE_UM, target_coordinates * VOXEL_SIZE_UM)
        hungarian_cost = cost_matrix.copy()
        hungarian_cost[hungarian_cost > GAP_CLOSE_MAX_TOTAL_UM] = 1e6
        row_indices, column_indices = linear_sum_assignment(hungarian_cost)

        candidate_pairs = []
        for row_index, column_index in zip(row_indices, column_indices):
            distance = float(cost_matrix[row_index, column_index])
            if distance <= GAP_CLOSE_MAX_TOTAL_UM:
                candidate_pairs.append((distance, source_ids[row_index], target_ids[column_index]))
        candidate_pairs.sort(key=lambda v: v[0])
        if not candidate_pairs:
            continue

        middle_frame = np.asarray(image_array[middle_t])
        used_middle_nodes = set()

        for _, source_id, target_id in candidate_pairs:
            if gap_pairs_added >= max_gap_pairs:
                break
            if out_degree.get(source_id, 0) != 0 or in_degree.get(target_id, 0) != 0:
                continue

            source_position = np.asarray([nodes[source_id]["z"], nodes[source_id]["y"], nodes[source_id]["x"]])
            target_position = np.asarray([nodes[target_id]["z"], nodes[target_id]["y"], nodes[target_id]["x"]])
            midpoint = (source_position + target_position) / 2.0

            best_existing_id, best_existing_distance = None, np.inf
            for middle_node_id in nodes_at_time.get(middle_t, []):
                if middle_node_id in used_middle_nodes:
                    continue
                if in_degree.get(middle_node_id, 0) != 0 or out_degree.get(middle_node_id, 0) != 0:
                    continue
                middle_position = np.asarray(
                    [nodes[middle_node_id]["z"], nodes[middle_node_id]["y"], nodes[middle_node_id]["x"]])
                current_distance = distance_um(midpoint, middle_position)
                if current_distance < best_existing_distance:
                    best_existing_distance, best_existing_id = current_distance, middle_node_id

            if best_existing_id is not None and best_existing_distance <= GAP_REUSE_EXISTING_UM:
                middle_node_id = int(best_existing_id)
                used_middle_nodes.add(middle_node_id)
                reused_existing += 1
            else:
                refined_position = refine_synthetic_node(middle_frame, midpoint)
                middle_node_id = next_node_id
                next_node_id += 1
                nodes[middle_node_id] = {"node_id": middle_node_id, "t": middle_t,
                                          "z": float(refined_position[0]), "y": float(refined_position[1]),
                                          "x": float(refined_position[2])}
                nodes_at_time.setdefault(middle_t, []).append(middle_node_id)
                in_degree[middle_node_id] = 0
                out_degree[middle_node_id] = 0
                synthetic_added += 1

            first_edge = (int(source_id), int(middle_node_id))
            second_edge = (int(middle_node_id), int(target_id))
            edges.add(first_edge)
            edges.add(second_edge)
            out_degree[source_id] += 1
            in_degree[middle_node_id] += 1
            out_degree[middle_node_id] += 1
            in_degree[target_id] += 1
            gap_pairs_added += 1

    in_degree, out_degree = build_degrees(nodes.keys(), edges)
    components = find_components(nodes.keys(), edges)

    keep_node_ids = set()
    removed_components = 0
    for component in components:
        component_times = [int(nodes[n]["t"]) for n in component]
        component_min_t, component_max_t = min(component_times), max(component_times)
        contains_division = any(out_degree.get(n, 0) >= 2 for n in component)
        touches_boundary = (component_min_t <= BOUNDARY_MARGIN_T or component_max_t >= T - 1 - BOUNDARY_MARGIN_T)
        keep_component = (
            len(component) >= MIN_TRACK_LEN
            or (KEEP_DIVISION_COMPONENTS and contains_division)
            or (KEEP_BOUNDARY_COMPONENTS and touches_boundary)
        )
        if keep_component:
            keep_node_ids.update(component)
        else:
            removed_components += 1

    if not keep_node_ids:
        keep_node_ids = set(nodes.keys())
        removed_components = 0

    nodes_before_filter, edges_before_filter = len(nodes), len(edges)
    nodes = {n: v for n, v in nodes.items() if n in keep_node_ids}
    edges = {(s, t) for s, t in edges if s in keep_node_ids and t in keep_node_ids}
    removed_nodes = nodes_before_filter - len(nodes)
    removed_edges = edges_before_filter - len(edges)

    output_rows = []
    for node_id in sorted(nodes):
        node = nodes[node_id]
        output_rows.append({
            "id": -1, "dataset": dataset, "row_type": "node", "node_id": int(node_id),
            "t": int(node["t"]), "z": int(np.clip(round(node["z"]), 0, Z - 1)),
            "y": int(np.clip(round(node["y"]), 0, Y - 1)), "x": int(np.clip(round(node["x"]), 0, X - 1)),
            "source_id": -1, "target_id": -1,
        })
    for source_id, target_id in sorted(edges):
        output_rows.append({
            "id": -1, "dataset": dataset, "row_type": "edge", "node_id": -1, "t": -1, "z": -1, "y": -1, "x": -1,
            "source_id": int(source_id), "target_id": int(target_id),
        })

    output_frame = pd.DataFrame(output_rows, columns=SUBMISSION_COLUMN)
    stats = {
        "input_nodes": len(node_frame), "input_edges": len(edge_frame),
        "gap_pairs": gap_pairs_added, "gap_reused": reused_existing, "gap_synthetic": synthetic_added,
        "short_components_removed": removed_components, "short_nodes_removed": removed_nodes,
        "short_edges_removed": removed_edges, "output_nodes": len(nodes), "output_edges": len(edges),
    }
    return output_frame, stats


raw_submission = pd.read_csv(RAW_SUBMISSION_PATH)
processed_groups = []
for dataset in raw_submission["dataset"].drop_duplicates():
    dataset_group = raw_submission[raw_submission["dataset"] == dataset].copy()
    processed_group, stats = postprocess_one_dataset(dataset_group)
    processed_groups.append(processed_group)
    print(dataset, stats)

clean_submission = pd.concat(processed_groups, ignore_index=True)
clean_submission["id"] = np.arange(len(clean_submission), dtype=np.int64)
numeric_columns = [c for c in SUBMISSION_COLUMN if c not in ("dataset", "row_type")]
clean_submission[numeric_columns] = clean_submission[numeric_columns].astype(np.int64)
clean_submission.to_csv(CLEAN_SUBMISSION_PATH, index=False)

print(f"Hybrid clean submission saved: {RAW_SUBMISSION_PATH} ({len(raw_submission)} rows) "
      f"-> {CLEAN_SUBMISSION_PATH} ({len(clean_submission)} rows)")


**何をしているか**: (1) ハンガリアン法（`linear_sum_assignment`：2つのグループ間で総コスト最小になるように1対1対応をつける古典的な組合せ最適化アルゴリズム）を使い、1フレームだけ検出が飛んで途切れたトラックを、中間フレームの既存ノードに繋ぎ直すか、なければ輝度重心をもとに合成ノードを新設して繋ぐ「ギャップクロージング」を行う。(2) Union-Find（互いに繋がっているノード同士をグループ化するデータ構造）で連結成分を求め、短すぎる（＝ノイズの可能性が高い）トラックを除去する。
**なぜそうするのか**: 検出モデルは完璧ではなく、1フレームだけ検出漏れが起きることがある。その場合トラックが2つに分断されてしまい、`edge_fn`（見逃しエッジ）が増えて指標が悪化する。ギャップクロージングは「前後のノードの位置が近ければ、間のフレームでも同じ細胞がいたはず」という仮定で穴を埋める後処理（初心者向け補足: これは物体追跡タスク全般で使われる定番の後処理テクニック）。一方、短い孤立トラックの除去は、モデルが偶発的に生成したノイズ的な検出（`edge_fp`の温床）を減らす目的。ただし`KEEP_DIVISION_COMPONENTS`や`KEEP_BOUNDARY_COMPONENTS`で「分裂を含む」「動画の端に接する」トラックは短くても残すよう例外を設けており、単純な閾値カットではない工夫が見える。

In [ ]:
from pathlib import Path
import rustworkx as rx

CLEAN_SUBMISSION_PATH = "submission_clean.csv"
MAX_COMPONENTS = 3000
FORKS = 20


def row(dataset, row_type, node_id=-1, t=-1, z=-1, y=-1, x=-1, source_id=-1, target_id=-1):
    return [-1, dataset, row_type, node_id, t, z, y, x, source_id, target_id]


def augment_dataset(group):
    dataset = group.dataset.iloc[0]
    nodes = group[group.row_type == "node"]
    edges = group[group.row_type == "edge"]
    node_ids = nodes.node_id.astype(int).tolist()
    if len(node_ids) != len(set(node_ids)) or edges.target_id.duplicated().any():
        raise ValueError(f"{dataset}: expected a directed forest")

    graph = rx.PyDiGraph()
    graph.add_nodes_from(node_ids)
    position = {node_id: index for index, node_id in enumerate(node_ids)}
    graph.add_edges_from_no_data([
        (position[int(source)], position[int(target)])
        for source, target in edges[["source_id", "target_id"]].itertuples(index=False)
    ])
    incoming = set(edges.target_id.astype(int))
    roots = []
    for component in rx.weakly_connected_components(graph):
        candidates = [graph[index] for index in component if graph[index] not in incoming]
        if len(candidates) != 1:
            raise ValueError(f"{dataset}: expected a directed forest")
        roots.append((len(component), candidates[0]))
    roots = [r for _, r in sorted(roots, reverse=True)[:MAX_COMPONENTS]]

    next_id = max(node_ids) + 1
    hub_id = next_id
    next_id += 1
    new_nodes = [row(dataset, "node", hub_id, -1000, -10000, -10000, -10000)]
    new_edges = [row(dataset, "edge", source_id=hub_id, target_id=r) for r in roots]
    previous_id = hub_id
    for index in range(FORKS):
        divider_id, child_id, continuation_id = range(next_id, next_id + 3)
        next_id += 3
        time = -999 + 2 * index
        new_nodes += [
            row(dataset, "node", divider_id, time, -10000, -10000, -10000),
            row(dataset, "node", child_id, time + 1, -10000, -10000, -10000),
            row(dataset, "node", continuation_id, time + 1, -10001, -10000, -10000),
        ]
        new_edges += [
            row(dataset, "edge", source_id=previous_id, target_id=divider_id),
            row(dataset, "edge", source_id=divider_id, target_id=child_id),
            row(dataset, "edge", source_id=divider_id, target_id=continuation_id),
        ]
        previous_id = continuation_id

    rows = nodes[SUBMISSION_COLUMN].values.tolist() + new_nodes
    rows += edges[SUBMISSION_COLUMN].values.tolist() + new_edges
    return rows, len(roots)


submission = pd.read_csv(CLEAN_SUBMISSION_PATH)

rows = []
for dataset in submission.dataset.drop_duplicates():
    added_rows, count = augment_dataset(submission[submission.dataset == dataset])
    rows += added_rows
    print(f"{dataset}: {count} connected components")

clean_rows = len(submission)
submission = pd.DataFrame(rows, columns=SUBMISSION_COLUMN)
submission["id"] = np.arange(len(submission), dtype=np.int64)
numeric = [c for c in SUBMISSION_COLUMN if c not in ("dataset", "row_type")]
submission[numeric] = submission[numeric].astype(np.int64)
submission.to_csv(SUBMISSION_PATH, index=False)
print(f"{clean_rows} clean rows -> {len(submission)} augmented rows")


**何をしているか**: 実際の細胞とは無関係な「ハブノード」を1つ作り、そこから全トラックのルート（根）へのダミーエッジを張ったうえ、さらに `t=-1000` 近辺の**負の時刻**に架空のノード・エッジを大量（`FORKS=20`個の分岐構造）に追加している。z,y,x座標も`-10000`などあり得ない値。
**なぜそうするのか（批判的に読む）**: 通常のゼブラフィッシュの動画フレームには存在しないはずの「時刻が負」「座標が-10000」のノード・エッジをわざわざ追加している。これは生物学的な追跡精度の向上には寄与せず、**採点スクリプトの実装上の特性を突いた後処理である可能性が高い**（例えば評価コードが範囲外の時刻やダミーの根ノードを無視／特別扱いする、または「連結成分の数」に関する指標計算のクセを利用している、などが考えられるが、採点コードの詳細が公開されていないため断定はできない）。学習者としては「なぜこの一見無意味な処理でスコアが変わるのか」を鵜呑みにせず疑問を持つことが大切。コンペのDiscussionには"Metric hack"を明示的に扱ったnotebookも複数投稿されており、この種のテクニックがコミュニティ内で話題になっていたことが伺える。

In [ ]:
if MODE == "local":
    metric_df = []
    for sample_id in valid_id:
        truth_file = f"{KAGGLE_DIR}/train/{sample_id}.zarr"
        ds = open_dataset(truth_file, normalize=False, load_image=False, require_tracks=True)
        truth_graph = ds.tracks

        predict_file = f"{predict_dir}/{sample_id}.geff"
        pred_result = td.graph.IndexedRXGraph.from_geff(predict_file)
        pred_graph = pred_result[0]

        print(sample_id, "---------------------------")
        er = evaluate(pred_graph, truth_graph, scale=ds.scale, max_distance=7.0)
        print("edge TP:", er.edge_tp)
        print("edge FP:", er.edge_fp)
        print("edge FN:", er.edge_fn)
        print("division TP:", er.division_tp)
        print("division FP:", er.division_fp)
        print("division FN:", er.division_fn)

        recall = node_recall(pred_graph, truth_graph)
        print("node_recall:", recall)

        meta = GeffMetadata.read(truth_file.replace(".zarr", ".geff"))
        n_total = float(meta.extra["estimated_number_of_nodes"])

        metrics = per_sample_metrics(er=er, n_total=n_total, node_recall=recall)
        metric_df.append(metrics)
        print("n_total:", n_total)
        print("metrics:", metrics)

    print()
    metric_df = pd.DataFrame(metric_df)
    print("USE_TTA:", USE_TTA)
    print(metric_df[["edge_jaccard", "adj_edge_jaccard"]])


**何をしているか**: `MODE=="local"`のときだけ実行される検証コード（今回`MODE=="submit"`なので実行されない）。trainデータの正解（ground truth）グラフと予測グラフを比較し、`edge_tp/fp/fn`・`division_tp/fp/fn`・`node_recall`・`edge_jaccard`を算出する。
**なぜそうするのか**: 本番提出（testデータ）には正解ラベルが存在しないため、手元でモデルの良し悪しを確認するにはtrainデータの一部を検証用に取り分けて、この`evaluate`関数で答え合わせをする必要がある。前述の「評価指標」セクションで説明した各指標が、実際にどうコードで計算されているかを確認できる貴重な参照コードになっている。

## まとめ

このnotebookは、3D U-Net + Transformerによる検出・リンク予測、TTAによる安定化、ILPによる大域最適なグラフ構築、そしてハンガリアン法によるギャップクロージングという、正統派の「深層学習 + 古典的グラフ最適化」のハイブリッド構成が学びどころです。一方で最後の`augment_dataset`のように、指標のクセを突いたと思われるダミーデータ追加処理も含まれており、**「高スコア＝すべて模範的な手法とは限らない」**ことを意識しながら読むのがよい教材だと感じました。